### Split adata from our cardiac multiome atlas into multiome, ATAC-only, and RNA-only
### This will allow us to perform scE2G analysis

In [1]:
import gzip
import os
import tempfile
from pathlib import Path
import numpy as np
import pooch
import scanpy as sc
import scvi
import seaborn as sns
import torch
from scipy.sparse import hstack
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import gc
import snapatac2 as snap

### Load in the 10X multiome barcode correspondence file

In [2]:
def reverse_complement(sequence):
    '''
    Return the reverse complement of a DNA sequence. 
    Characters after a '-' are copied as-is and not included in reverse complement processing.
    This is needed in order to get the ATAC barcodes to match up with the RNA barcodes.
    '''
    complement = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

    if '-' in sequence:
        dna_sequence, suffix = sequence.split('-', 1)
    else:
        dna_sequence, suffix = sequence, ''

    # Get the reverse complement of the DNA sequence
    reverse_comp = ''.join(complement[base] for base in reversed(dna_sequence))

    # Return the reverse complement and append the suffix
    return reverse_comp + ('-' + suffix if suffix else '')

In [6]:
multiome_barcode_df = pd.read_csv("../../../../Final_manuscript_analysis/ATAC/aggregated_analysis/multiome_mapping_files/ATAC_RNA_737K_barcodes.txt", index_col = 0)
multiome_barcode_df.head()

,RNA_barcode,ATAC_barcode
0,AAACAGCCAAACAACA,ACAGCGGGTGTGTTAC
1,AAACAGCCAAACATAG,ACAGCGGGTTGTTCTT
2,AAACAGCCAAACCCTA,ACAGCGGGTAACAGGC
3,AAACAGCCAAACCTAT,ACAGCGGGTGCGCGAA
4,AAACAGCCAAACCTTG,ACAGCGGGTCCTCCAT


### Load in the RNA and ATAC

In [8]:
%%time
RNA_adata = sc.read_h5ad("../../../../Final_manuscript_analysis/RNA/aggregated_analysis/07_final_RNA_without_scvi.h5ad")
# use the raw counts
RNA_adata.X = RNA_adata.layers['counts']
RNA_adata

CPU times: user 17.3 s, sys: 2min 7s, total: 2min 24s
Wall time: 7min 1s


AnnData object with n_obs × n_vars = 2305964 × 16115
    obs: 'age', 'donor_id', 'sex', 'region', 'cell_type', 'disease', 'consistent_cell_type', 'study', 'technology', 'cell_or_nuclei', 'barcode', 'sample_id', 'age_status', 'tech_plus_study', 'disease_binary', 'decade', 'age_group', '_scvi_batch', '_scvi_labels', 'leiden_scVI', 'scvi_cell_type', 'redo_leiden_0.5', 'UMAP1', 'UMAP2', 'v2_scvi_cell_type', 'final_cell_type'
    obsm: 'X_scVI', 'X_umap', '_scvi_extra_categorical_covs'
    layers: 'counts'

In [9]:
%%time
# use the 500 nt peaks, not the 5kb tiles 
ATAC_adata = sc.read_h5ad("../../../../Final_manuscript_analysis/ATAC/aggregated_analysis/bigwig_bedgraph/final_adata_with_fragments.h5ad")

CPU times: user 16.1 s, sys: 2min 41s, total: 2min 57s
Wall time: 3min 34s


In [10]:
ATAC_adata

AnnData object with n_obs × n_vars = 690044 × 606219
    obs: 'ATAC_barcode', 'sample_id', 'leiden', 'donor_id', 'study', 'age_status', 'age', 'sex', 'region', 'disease_binary', 'technology', 'fragment_file', 'full_path', 'file', 'nfrag', 'tsse', 'cell_type', 'tech_plus_study'
    var: 'count', 'selected'
    uns: 'AnnDataSet', 'macs3', 'reference_sequences', 'spectral_eigenvalue'
    obsm: 'X_spectral', 'X_spectral_harmony', 'X_umap', 'fragment_paired'
    obsp: 'distances'

### Determine the number of features

In [11]:
num_genes = RNA_adata.shape[1]
num_tiles = ATAC_adata.shape[1]
print(f"There are {num_genes} genes and {num_tiles} peaks")

There are 16115 genes and 606219 peaks


### First construct the shared multiome object

We will again split the multiome into the two studies, as in `04B_annotate_leiden_clusters.ipynb`, as there are differences between how the ATAC and RNA correspond to each other. 

In [ ]:
%%time
Kanemaru_ATAC_adata = ATAC_adata[ATAC_adata.obs.study == "Kanemaru 2023"].copy()
Kanemaru_RNA_adata = RNA_adata[RNA_adata.obs.study == "Kanemaru 2023"].copy()

print(Kanemaru_ATAC_adata.shape)
print(Kanemaru_RNA_adata.shape)

Add back the `adata.obsm[fragment_paired]`, which are lost when subsetting

In [ ]:
%%time
Kanemaru_ATAC_adata.obsm["fragment_paired"] = ATAC_adata.obsm["fragment_paired"][ATAC_adata.obs["study"] == "Kanemaru 2023", :]

In [ ]:
Kanemaru_ATAC_adata

We need to join on `sample_id` rather than `donor_id` for Kanemaru samples, as some donors had multiple runs

In [ ]:
Kanemaru_ATAC_adata.obs = Kanemaru_ATAC_adata.obs.reset_index()

In [ ]:
# remove the sample_id, which is before ":"
# in the case of the Kanemaru samples, the barcode in the fragment files is actually already the RNA barcode from the 10X 737K whitelist

Kanemaru_ATAC_adata.obs['RNA_barcode'] = Kanemaru_ATAC_adata.obs['barcode'].str.split(":").str[1]

# also create a unique multiome barcode for these data
Kanemaru_ATAC_adata.obs['multiome_barcode'] = ( Kanemaru_ATAC_adata.obs['sample_id'].astype(str) + ":" + 
                                               Kanemaru_ATAC_adata.obs['RNA_barcode'].astype(str)  )

# for the RNA object, also do this
Kanemaru_RNA_adata.obs['RNA_barcode'] = Kanemaru_RNA_adata.obs_names.str.split(":").str[1]

# also create a unique multiome barcode for these data
Kanemaru_RNA_adata.obs['multiome_barcode'] = ( Kanemaru_RNA_adata.obs['sample_id'].astype(str) + ":" + 
                                               Kanemaru_RNA_adata.obs['RNA_barcode'].astype(str)  )

In [ ]:
Kanemaru_ATAC_metadata = Kanemaru_ATAC_adata.obs
Kanemaru_RNA_metadata = Kanemaru_RNA_adata.obs
merged_Kanemaru_df = Kanemaru_ATAC_metadata.merge(Kanemaru_RNA_metadata, on = ["multiome_barcode"])

print(merged_Kanemaru_df.shape)
intersecting_Kanemaru_multiome_barcodes = merged_Kanemaru_df.multiome_barcode

In [ ]:
# confirm that these are all unique
Counter(merged_Kanemaru_df.multiome_barcode).most_common()[0]

In [ ]:
# filter the RNA Kanemaru adata to just those with multiome_barcodes

filt_Kanemaru_RNA_adata = Kanemaru_RNA_adata
filt_Kanemaru_RNA_adata.obs_names = filt_Kanemaru_RNA_adata.obs['multiome_barcode']
filt_Kanemaru_RNA_adata = filt_Kanemaru_RNA_adata[intersecting_Kanemaru_multiome_barcodes, :]
print(filt_Kanemaru_RNA_adata)

filt_Kanemaru_ATAC_adata = Kanemaru_ATAC_adata
filt_Kanemaru_ATAC_adata.obs_names = filt_Kanemaru_ATAC_adata.obs['multiome_barcode']
filt_Kanemaru_ATAC_adata = filt_Kanemaru_ATAC_adata[intersecting_Kanemaru_multiome_barcodes, :]
print(filt_Kanemaru_ATAC_adata)

In [ ]:
def create_multiome_object(RNA_adata, ATAC_adata):
    '''Combine two adata objects, RNA and ATAC that must have the same order of barcodes in adata.obs_names into
    an adata object with both modalities present
    '''
    
    # check that the order of the barcodes is identical 
    assert np.array_equal(RNA_adata.obs_names, ATAC_adata.obs_names)

    # combine the count information for RNA and ATAC
    merged_X = hstack([RNA_adata.X, ATAC_adata.X])

    # combine the variable (genes and peaks) from the original adatas
    merged_var = pd.concat([RNA_adata.var, ATAC_adata.var], axis=0)

    # create new merged adata using the merged counts and variables; used the adata from the RNA adata
    merged_adata = sc.AnnData(X=merged_X, obs=RNA_adata.obs.copy(), var=merged_var)

    # format adata.var to be compatible with MultiVI
    merged_adata.var['modality'] = "Gene Expression"
    merged_adata.var.loc[merged_adata.var_names.str.startswith("chr"), 'modality'] = "Peaks"

    return(merged_adata)

In [ ]:
%%time
Kanemaru_multiome_adata = create_multiome_object(RNA_adata = filt_Kanemaru_RNA_adata, ATAC_adata = filt_Kanemaru_ATAC_adata)

In [ ]:
Kanemaru_multiome_adata

In [ ]:
%%time
filt_Kanemaru_ATAC_adata.write("01_filtered_Kanemaru_multiome_ATAC_adata.h5ad")

In [ ]:
filt_Kanemaru_ATAC_adata

In [ ]:
%%time
filt_Kanemaru_RNA_adata.write("01_filtered_Kanemaru_multiome_RNA_adata.h5ad")

In [25]:
%%time
#Kanemaru_multiome_adata.write("01_filtered_Kanemaru_multiome_both_adata.h5ad")

CPU times: user 1e+03 ns, sys: 5 μs, total: 6 μs
Wall time: 11.9 μs


In [26]:
Kanemaru_multiome_adata.obs

,age,donor_id,sex,region,cell_type,disease,consistent_cell_type,study,technology,cell_or_nuclei,...,_scvi_labels,leiden_scVI,scvi_cell_type,redo_leiden_0.5,UMAP1,UMAP2,v2_scvi_cell_type,final_cell_type,RNA_barcode,multiome_barcode
multiome_barcode,,,,,,,,,,,,,,,,,,,,,
HCAHeartST10773166_HCAHeartST10781063:AATTCGTCAACAGCCT-1,47.5,Kanemaru 2023:AH1-Nuclei_Multiome-v1,female,LV,Ventricular Cardiomyocyte,ND,Cardiomyocyte,Kanemaru 2023,Multiome-v1,Nuclei,...,0,1,Cardiomyocyte,4,13.412882,3.175658,Cardiomyocyte,Cardiomyocyte,AATTCGTCAACAGCCT-1,HCAHeartST10773166_HCAHeartST10781063:AATTCGTC...
HCAHeartST11064575_HCAHeartST11023240:GTTTCTAGTTGCTGGG-1,47.5,Kanemaru 2023:AH1-Nuclei_Multiome-v1,female,LV,Mural cell,ND,Pericyte,Kanemaru 2023,Multiome-v1,Nuclei,...,0,9,Pericyte,0,1.942001,-6.066140,Pericyte,Pericyte,GTTTCTAGTTGCTGGG-1,HCAHeartST11064575_HCAHeartST11023240:GTTTCTAG...
HCAHeartST11064575_HCAHeartST11023240:CTTTGGGAGCTAATTG-1,47.5,Kanemaru 2023:AH1-Nuclei_Multiome-v1,female,LV,Endothelial cell,ND,Endothelial,Kanemaru 2023,Multiome-v1,Nuclei,...,0,18,Endothelial,8,-2.079240,6.921587,Endothelial,Endothelial,CTTTGGGAGCTAATTG-1,HCAHeartST11064575_HCAHeartST11023240:CTTTGGGA...
HCAHeart9508629_HCAHeart9508821:CATTGTAAGTTTCCTG-1,62.5,Kanemaru 2023:D7-Nuclei_Multiome-v1,male,LV,Ventricular Cardiomyocyte,ND,Cardiomyocyte,Kanemaru 2023,Multiome-v1,Nuclei,...,0,1,Cardiomyocyte,5,11.385880,2.086848,Cardiomyocyte,Cardiomyocyte,CATTGTAAGTTTCCTG-1,HCAHeart9508629_HCAHeart9508821:CATTGTAAGTTTCC...
HCAHeartST11064575_HCAHeartST11023240:ATCGCCCGTTGGCCGA-1,47.5,Kanemaru 2023:AH1-Nuclei_Multiome-v1,female,LV,Endothelial cell,ND,Endothelial,Kanemaru 2023,Multiome-v1,Nuclei,...,0,18,Endothelial,7,-1.628683,7.780728,Endothelial,Endothelial,ATCGCCCGTTGGCCGA-1,HCAHeartST11064575_HCAHeartST11023240:ATCGCCCG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
HCAHeartST10773166_HCAHeartST10781063:TTTCTCACATAAGTTC-1,47.5,Kanemaru 2023:AH1-Nuclei_Multiome-v1,female,LV,Ventricular Cardiomyocyte,ND,Cardiomyocyte,Kanemaru 2023,Multiome-v1,Nuclei,...,0,11,Cardiomyocyte,6,14.755566,2.667149,Cardiomyocyte,Cardiomyocyte,TTTCTCACATAAGTTC-1,HCAHeartST10773166_HCAHeartST10781063:TTTCTCAC...
HCAHeart9508627_HCAHeart9508819:CCTGATGAGGACCGCT-1,57.5,Kanemaru 2023:D3-Nuclei_Multiome-v1,male,LV,Myeloid,ND,Myeloid,Kanemaru 2023,Multiome-v1,Nuclei,...,0,14,Myeloid,8,1.514995,15.332829,Myeloid,Myeloid,CCTGATGAGGACCGCT-1,HCAHeart9508627_HCAHeart9508819:CCTGATGAGGACCG...
HCAHeart9845431_HCAHeart9917173:CAATCCTGTTTAAAGC-1,47.5,Kanemaru 2023:D8-Nuclei_Multiome-v1,male,LV,Ventricular Cardiomyocyte,ND,Cardiomyocyte,Kanemaru 2023,Multiome-v1,Nuclei,...,0,1,Cardiomyocyte,4,13.838859,0.494169,Cardiomyocyte,Cardiomyocyte,CAATCCTGTTTAAAGC-1,HCAHeart9845431_HCAHeart9917173:CAATCCTGTTTAAA...


### Perform the same for the ENCODE data (much larger)

In [27]:
%%time
ENCODE_ATAC_adata = ATAC_adata[ATAC_adata.obs.study == "ENCODE v4 (Snyder)"].copy()
ENCODE_RNA_adata = RNA_adata[RNA_adata.obs.study == "ENCODE v4 (Snyder)"].copy()

print(ENCODE_ATAC_adata.shape)
print(ENCODE_RNA_adata.shape)

(494790, 606219)
(484348, 16115)
CPU times: user 1min 20s, sys: 12min 14s, total: 13min 34s
Wall time: 13min 42s


In [28]:
ENCODE_metadata = pd.read_csv("../../Final_manuscript_analysis/ATAC/aggregated_analysis/01C_QC_updated_metadata.csv", index_col=0)

In [29]:
ENCODE_multiome_sample_ids = ENCODE_metadata[(ENCODE_metadata['technology'] == "10X_Multiome") & (ENCODE_metadata['study'] == "ENCODE v4 (Snyder)")]['sample_id'].unique()

In [30]:
%%time
# filter to just multiome
ENCODE_RNA_adata = ENCODE_RNA_adata[ENCODE_RNA_adata.obs.technology == "Multiome-v1"]
print(ENCODE_RNA_adata.shape)

(484348, 16115)
CPU times: user 283 ms, sys: 15.6 ms, total: 299 ms
Wall time: 297 ms


In [31]:
%%time
ENCODE_ATAC_adata = ENCODE_ATAC_adata[ENCODE_ATAC_adata.obs.sample_id.isin(ENCODE_multiome_sample_ids)]
print(ENCODE_ATAC_adata.shape)

(466381, 606219)
CPU times: user 291 ms, sys: 20.9 ms, total: 312 ms
Wall time: 310 ms


In [32]:
ENCODE_ATAC_adata.obs = ENCODE_ATAC_adata.obs.reset_index()

In [33]:
%%time 
# find the corresponding RNA barcode for the ENCODE ATAC multiome
ENCODE_ATAC_adata.obs['ATAC_barcode'] = ENCODE_ATAC_adata.obs['barcode'].str.split(":").str[1]
# take the reverse complement
ENCODE_ATAC_adata.obs['ATAC_barcode'] = ENCODE_ATAC_adata.obs['ATAC_barcode'].apply(lambda x: reverse_complement(x))

CPU times: user 3.11 s, sys: 4.64 s, total: 7.75 s
Wall time: 7.73 s


In [34]:
# merge on the multiome df, this will add the RNA barcode
ENCODE_ATAC_adata.obs = ENCODE_ATAC_adata.obs.merge(multiome_barcode_df, on = "ATAC_barcode", how="left")

In [35]:
%%time
ENCODE_ATAC_adata = ENCODE_ATAC_adata[~ENCODE_ATAC_adata.obs["RNA_barcode"].isna()].copy()

/home/william/.local/lib/python3.12/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


CPU times: user 59.6 s, sys: 2min 42s, total: 3min 42s
Wall time: 3min 43s


In [36]:
len (set(ENCODE_ATAC_adata.obs.RNA_barcode) & set(ENCODE_RNA_adata.obs.barcode) )

253876

In [37]:
len (set(ENCODE_ATAC_adata.obs.donor_id) & set(ENCODE_RNA_adata.obs.donor_id) )

62

The `sample_id` field for ATAC and RNA is different for ENCODE, while the `donor_id` field is the same

In [38]:
ENCODE_ATAC_adata.obs['multiome_barcode'] = ( ENCODE_ATAC_adata.obs['donor_id'].astype(str) + ":" + 
                                             ENCODE_ATAC_adata.obs['RNA_barcode'].astype(str) )

In [39]:
%%time

# do the same for the RNA portion
ENCODE_RNA_adata.obs["multiome_barcode"] = ( ENCODE_RNA_adata.obs["donor_id"].astype(str) + ":" + 
                                             ENCODE_RNA_adata.obs["barcode"].astype(str) )

<timed exec>:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.


CPU times: user 7.76 s, sys: 10.4 s, total: 18.2 s
Wall time: 18.2 s


In [40]:
ENCODE_multiome_barcodes =  list ( set(ENCODE_RNA_adata.obs.multiome_barcode) & set(ENCODE_ATAC_adata.obs.multiome_barcode) )
print(f"Number of intersecting nuclei with both ATAC and RNA modalities: {len(ENCODE_multiome_barcodes)}")

Number of intersecting nuclei with both ATAC and RNA modalities: 287596


#### Filter the ENCODE ATAC and RNA only to the intersecting multiomes

In [41]:
%%time 
filt_ENCODE_ATAC_adata = ENCODE_ATAC_adata
filt_ENCODE_ATAC_adata.obs_names = filt_ENCODE_ATAC_adata.obs.multiome_barcode
filt_ENCODE_ATAC_adata = filt_ENCODE_ATAC_adata[filt_ENCODE_ATAC_adata.obs_names.isin(ENCODE_multiome_barcodes)].copy()
filt_ENCODE_ATAC_adata = filt_ENCODE_ATAC_adata[ENCODE_multiome_barcodes, :]

CPU times: user 31.6 s, sys: 5min 38s, total: 6min 10s
Wall time: 6min 14s


In [55]:
%%time
filt_ENCODE_ATAC_adata.write("01_filtered_ENCODE_multiome_ATAC_adata.h5ad")

/home/william/.local/lib/python3.12/site-packages/anndata/_core/anndata.py:1209: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'ATAC_barcode' as categorical
/home/william/.local/lib/python3.12/site-packages/anndata/_core/anndata.py:1209: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'RNA_barcode' as categorical


CPU times: user 1min 12s, sys: 14min 35s, total: 15min 48s
Wall time: 16min 1s


In [42]:
%%time
filt_ENCODE_RNA_adata = ENCODE_RNA_adata
filt_ENCODE_RNA_adata.obs_names = filt_ENCODE_RNA_adata.obs.multiome_barcode
filt_ENCODE_RNA_adata = filt_ENCODE_RNA_adata[filt_ENCODE_RNA_adata.obs_names.isin(ENCODE_multiome_barcodes)].copy()
filt_ENCODE_RNA_adata = filt_ENCODE_RNA_adata[ENCODE_multiome_barcodes, :]

CPU times: user 5.7 s, sys: 35.9 s, total: 41.6 s
Wall time: 41.7 s


In [56]:
%%time
filt_ENCODE_RNA_adata.write("01_filtered_ENCODE_multiome_RNA_adata.h5ad")

CPU times: user 6.86 s, sys: 1min 48s, total: 1min 55s
Wall time: 1min 57s


#### For the filtered ATAC, add back the fragment files

In [53]:
%%time
# create multiome adata for MultiVI
#ENCODE_multiome_adata = create_multiome_object(RNA_adata = filt_ENCODE_RNA_adata, 
#                                               ATAC_adata = filt_ENCODE_ATAC_adata)

CPU times: user 6 μs, sys: 11 μs, total: 17 μs
Wall time: 33.4 μs


### Combine the multiome adata together

In [ ]:
%%time
all_multiome_adata = sc.concat([ENCODE_multiome_adata, Kanemaru_multiome_adata])
all_multiome_adata

In [ ]:
all_multiome_adata.shape[0] - ENCODE_multiome_adata.shape[0] - Kanemaru_multiome_adata.shape[0]

#### The multiome dataset has 356,754 nuclei

### Get the ATAC and RNA adata without the multiome barcodes

#### Extract the non-multiome RNA modality only nuclei 

In [ ]:
# for some of the donors, make the multiome_barcode = donor_id + barcode
RNA_adata.obs['multiome_barcode'] = RNA_adata.obs['donor_id'].astype(str) + ":" + RNA_adata.obs['barcode'].astype(str)

In [ ]:
# however, for the Kanemaru, make multiome_barcode = sample_id + barcode
RNA_adata.obs.loc[RNA_adata.obs['study'] == "Kanemaru 2023", 'multiome_barcode'] = (
    RNA_adata.obs.loc[RNA_adata.obs['study'] == "Kanemaru 2023", 'sample_id'].astype(str) + ":" +
    RNA_adata.obs.loc[RNA_adata.obs['study'] == "Kanemaru 2023", 'barcode'].astype(str)
)

In [ ]:
%%time
non_multiome_RNA_adata = RNA_adata[~RNA_adata.obs.multiome_barcode.isin(all_multiome_adata.obs_names)]
non_multiome_RNA_adata
print(RNA_adata.shape[0] - non_multiome_RNA_adata.shape[0])

Indeed, this non_multiome_RNA now doesn't have exactly 356,754 nuclei, which are multiome

#### Extract the non-multiome ATAC modality only nuclei

In [ ]:
ATAC_adata.obs = ATAC_adata.obs.reset_index()

In [ ]:
# extract just the barcode
ATAC_adata.obs['ATAC_barcode'] = ATAC_adata.obs['barcode'].str.split(":").str[1]
# perform reverse complement only for the ENCODE samples
ATAC_adata.obs.loc[ATAC_adata.obs['study'] == "ENCODE v4 (Snyder)", 'ATAC_barcode'] = (
    ATAC_adata.obs.loc[ATAC_adata.obs['study'] == "ENCODE v4 (Snyder)", 'ATAC_barcode']
    .apply(reverse_complement)
)

# merge on the 737K mapping to get the RNA barcode; there will only be a match for the ENCODE samples
ATAC_adata.obs = ATAC_adata.obs.merge(multiome_barcode_df, how="left", on = "ATAC_barcode")

In [ ]:
# for the Kanemaru samples, they don't use this matching, so such make the "RNA_barcode" equal to the "ATAC_barcode"
ATAC_adata.obs.loc[ATAC_adata.obs.study == "Kanemaru 2023", 'RNA_barcode'] = ATAC_adata.obs.loc[ATAC_adata.obs.study == "Kanemaru 2023", 'ATAC_barcode']

In [ ]:
# add the multiome_barcode as donor_id + RNA_barcode, except for Kanemaru 2023, for which we will use the sample id
ATAC_adata.obs['multiome_barcode'] = ATAC_adata.obs['donor_id'].astype(str) + ":" + ATAC_adata.obs['RNA_barcode'].astype(str)

In [ ]:
# however, for the Kanemaru, make multiome_barcode = sample_id + barcode
ATAC_adata.obs.loc[ATAC_adata.obs['study'] == "Kanemaru 2023", 'multiome_barcode'] = (
    ATAC_adata.obs.loc[ATAC_adata.obs['study'] == "Kanemaru 2023", 'sample_id'].astype(str) + ":" +
    ATAC_adata.obs.loc[ATAC_adata.obs['study'] == "Kanemaru 2023", 'RNA_barcode'].astype(str)
)

In [ ]:
%%time
non_multiome_ATAC_adata =  ( ATAC_adata[ (~ATAC_adata.obs.multiome_barcode.isin(all_multiome_adata.obs.multiome_barcode) ), :] )
non_multiome_ATAC_adata

In [ ]:
ATAC_adata.shape[0] - non_multiome_ATAC_adata.shape[0] - all_multiome_adata.shape[0]

In [ ]:
non_multiome_ATAC_adata.obs_names = non_multiome_ATAC_adata.obs.barcode

In [ ]:
print(f"Number of non-multiome ATAC nuclei {non_multiome_ATAC_adata.shape[0]}")

In [ ]:
print(f"Number of non-multiome RNA nuclei {non_multiome_RNA_adata.shape[0]}")

In [ ]:
print(f"Number of multiome nuclei: {all_multiome_adata.shape[0]}")

In [ ]:
print(f"Total number of distinct nuclei: {all_multiome_adata.shape[0] + non_multiome_ATAC_adata.shape[0] + non_multiome_RNA_adata.shape[0]}")

### Confirm that all of the obs_names are unique

In [ ]:
%%time
non_multiome_RNA_adata.obs.index = non_multiome_RNA_adata.obs_names
non_multiome_ATAC_adata.obs.index = non_multiome_ATAC_adata.obs_names
all_multiome_adata.obs.index = all_multiome_adata.obs_names

In [ ]:
set(non_multiome_ATAC_adata.obs.index) & set(non_multiome_RNA_adata.obs.index)

In [ ]:
set(all_multiome_adata.obs.index) & set(non_multiome_RNA_adata.obs.index) 

In [ ]:
set(all_multiome_adata.obs.index) & set(non_multiome_ATAC_adata.obs.index)

In [ ]:
%%time
non_multiome_ATAC_adata.write("01_non_multiome_ATAC_adata.h5ad")
non_multiome_ATAC_adata

In [ ]:
%%time
non_multiome_RNA_adata.write("01_non_multiome_RNA_adata.h5ad")
non_multiome_RNA_adata

In [ ]:
%%time
all_multiome_adata.write("01_multiome_ATAC_RNA_adata.h5ad")
all_multiome_adata

#### Split the ATAC/RNA for multiome

In [ ]:
multiome_ATAC = all_multiome_adata[:, all_multiome_adata.var_names.str.startswith("chr")].copy()

In [ ]:
multiome_ATAC

In [ ]:
multiome_ATAC.write("01_multiome_ATAC.h5ad")

In [ ]:
multiome_RNA = all_multiome_adata[:, ~all_multiome_adata.var_names.str.startswith("chr")].copy()

In [ ]:
multiome_RNA

In [ ]:
multiome_RNA.write("01_multiome_RNA.h5ad")